# Import libraries

In [ ]:
import all_functions
from all_functions import *

import cusum_functions
from cusum_functions import *
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# Load the causal graph

In [ ]:
edge_index = load_graph('.../edge_index.pt')
list_dir   = '.../list.txt'
var_list   = get_var_list(list_dir)

# Model deployment

In [ ]:
class Main():
    def __init__(self, train_config, env_config, debug=False):

        self.train_config = train_config
        self.env_config = env_config
        self.datestr = None

        train_orig = pd.read_csv('.../train.csv', sep=',', index_col=0)
        test_orig  = pd.read_csv('.../test.csv', sep=',', index_col=0)
       
        self.train = train_orig.copy()
        self.test  = test_orig.copy()
        
        if 'attack' in self.train.columns:
            train = self.train.drop(columns=['attack'])

        self.var_map = var_list
        
        set_device(env_config['device'])
        self.device = get_device()
        
        edge_index_sets = []
        self.edge_index = edge_index
        edge_index_sets.append(self.edge_index)
        
        train_dataset_indata = construct_data(self.train, var_list, labels=0)
        test_dataset_indata  = construct_data(self.test, var_list, labels=self.test.attack.tolist())

        cfg = {'slide_win': train_config['slide_win'],
               'slide_stride': train_config['slide_stride'],}

        train_dataset = TimeDataset(train_dataset_indata, self.edge_index, mode='train', config=cfg)
        test_dataset  = TimeDataset(test_dataset_indata, self.edge_index, mode='test', config=cfg)

        train_dataloader, val_dataloader = self.get_loaders(train_dataset, train_config['seed'], train_config['batch'], val_ratio = train_config['val_ratio'])
        
        self.train_dataset = train_dataset
        self.test_dataset = test_dataset
        
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader
        self.test_dataloader = DataLoader(test_dataset, batch_size=train_config['batch'], shuffle=False, num_workers=0)
        self.model = normalModel(edge_index_sets, 
                                 len(var_list), 
                                dim=train_config['dim'], 
                                input_dim=train_config['slide_win'],
                                out_layer_num=train_config['out_layer_num'],
                                out_layer_inter_dim=train_config['out_layer_inter_dim']).to(self.device)

    def get_save_path(self):

        dir_path = self.env_config['save_path']
        
        if self.datestr is None:
            now = datetime.now()
            self.datestr = now.strftime('%m-%d-%H_%M_%S')
        datestr = self.datestr          

        paths = [f'{dir_path}/pretrained/best_{datestr}.pt', 
                 f'{dir_path}/results/{datestr}.csv']

        for path in paths:
            dirname = os.path.dirname(path)
            Path(dirname).mkdir(parents=True, exist_ok=True)

        return paths

    def run(self, target_tick_ppr, q_lo, q_hi, max_iter, tol):
        if len(self.env_config['load_model_path']) > 0:
            model_save_path = self.env_config['load_model_path']
            
        else:
            model_save_path = self.get_save_path()[0]

            self.train_log = all_functions.train(self.model, model_save_path, 
                config = self.train_config,
                train_dataloader=self.train_dataloader,
                val_dataloader=self.val_dataloader,
                train_dataset=self.train_dataset)

        self.model.load_state_dict(torch.load(model_save_path, map_location=self.device))        
        best_model = self.model.to(self.device)
        
        _, self.test_result = test(best_model, self.test_dataloader)
        val_loss, self.val_result = test(best_model, self.val_dataloader)

        self.prepare_detection_inputs(self.test_result, self.val_result, target_tick_ppr, q_lo, q_hi, max_iter, tol)


    def get_loaders(self, train_dataset, seed, batch, val_ratio=0.1):
        dataset_len = int(len(train_dataset))
        train_use_len = int(dataset_len * (1 - val_ratio))
        val_use_len = int(dataset_len * val_ratio)
        val_start_index = random.randrange(train_use_len)
        indices = torch.arange(dataset_len)
        train_sub_indices = torch.cat([indices[:val_start_index], indices[val_start_index+val_use_len:]])
        val_sub_indices = indices[val_start_index:val_start_index+val_use_len]
        
        train_subset = Subset(train_dataset, train_sub_indices)
        val_subset = Subset(train_dataset, val_sub_indices)

        g =torch.Generator()
        g.manual_seed(seed)
        
        train_dataloader = DataLoader(train_subset, batch_size=batch, shuffle=True, generator=g, num_workers=0)
        val_dataloader = DataLoader(val_subset, batch_size=batch, shuffle=False, generator=g,  num_workers=0)

        return train_dataloader, val_dataloader

    def prepare_detection_inputs(self, test_result, val_result, target_tick_ppr, q_lo, q_hi, max_iter, tol):
        np_test_result = np.array(test_result)
        self.gt_labels = np_test_result[2, :, 0].tolist()

        self.test_pred_scores, self.val_pred_scores     = get_full_pred_errors(test_result, val_result)
        self.sensor_thresholds, self.pred_sensor_labels = get_sensor_anomality_quantile_target_ppr(self.gt_labels, test_result, val_result, target_tick_ppr,
                                                                                                   q_lo, q_hi, max_iter, tol)


    def evaluate_pointwise_detection(self, test_result, val_result, target_tick_ppr, q_lo, q_hi, max_iter, tol, k):
        result_info = get_test_performance_quantile_target_ppr(self.gt_labels, test_result, val_result, target_tick_ppr, 
                                                               q_lo, q_hi, max_iter, tol, k)
        self.result_info = result_info

        print(
            "========================= "
            "Result of Point-Wise Anomaly Detection "
            "========================="
        )
        print(f"Macro-F1 score: {result_info[0]}")
        print(f"MCC score: {result_info[1]}")
        print(result_info[2])

        return result_info
    

if __name__ == "__main__":
    
    sys.argv = [''] 
    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        pl.seed_everything(seed, workers=True)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        os.environ['PYTHONHASHSEED'] = str(seed)
        
    parser = argparse.ArgumentParser()
    parser.add_argument('-batch', help='batch size', type = int, default=16)
    parser.add_argument('-epoch', help='train epoch', type = int, default=100)
    parser.add_argument('-slide_win', help='slide window size', type = int, default=30)
    parser.add_argument('-dim', help='node dimension', type = int, default=64)
    parser.add_argument('-slide_stride', help='slide stride', type = int, default=1)
    parser.add_argument('-save_path', help='save trained model', type = str, default='')
    parser.add_argument('-device', help='cuda / cpu', type = str, default='cuda')
    parser.add_argument('-random_seed', help='random seed', type = int, default=42)
    parser.add_argument('-out_layer_num', help='output size ', type = int, default=1)
    parser.add_argument('-out_layer_inter_dim', help='hidden dim of forecast layer', type = int, default=256)
    parser.add_argument('-decay', help='decay', type = float, default=0)
    parser.add_argument('-val_ratio', help='validation ratio', type = float, default=0.1)
    parser.add_argument('-lr', help='learning rate', type = float, default=0.001)
    parser.add_argument('-load_model_path', help='load pretrained model path', type = str, default='')
    args = parser.parse_args()
    
    train_config = {
        'batch': args.batch,
        'epoch': args.epoch,
        'slide_win': args.slide_win,
        'dim': args.dim,
        'slide_stride': args.slide_stride,
        'seed': args.random_seed,
        'out_layer_num': args.out_layer_num,
        'out_layer_inter_dim': args.out_layer_inter_dim,
        'decay': args.decay,
        'val_ratio': args.val_ratio,
        'lr': args.lr}

    set_seed(train_config['seed'])

    env_config={
        'save_path': args.save_path,
        'device': args.device,
        'load_model_path': args.load_model_path
    }
    
    normal_main = Main(train_config, env_config, debug=False)
    normal_main.run(target_tick_ppr=0.09, q_lo=0.90, q_hi=0.9999, max_iter=25, tol=0.0001)

# Markov ARL-based CUSUM-Anomaly Detection

In [ ]:
# Take inputs as prediction error signals
test_errors = np.asarray(normal_main.test_pred_scores, dtype=float).T
validation_errors = np.asarray(normal_main.val_pred_scores, dtype=float).T

sensor_thresholds = np.asarray(normal_main.sensor_thresholds, dtype=float)
original_labels = np.asarray(normal_main.gt_labels, dtype=int,)


# Probabilistic models (normal & abnormal states)
mu0_mle, sigma_mle, mle_success = (fit_folded_normal_mle(validation_errors))
mu1_mle = (mu0_mle + sensor_thresholds)


# Calculate validation and test LLR increments (=log-likelihood evidences)
validation_llr = compute_llr(errors=validation_errors, mu0=mu0_mle,mu1=mu1_mle, sigma=sigma_mle)
test_llr       = compute_llr(errors=test_errors, mu0=mu0_mle, mu1=mu1_mle, sigma=sigma_mle)

# Validation CUSUM trajectory.
(_, _, validation_S, validation_G) = cusum_from_increments(increments=validation_llr, thresholds=None, burn_in=30)

# State-conditioned Markov ARL calibration
(G_thresholds, lower_bounds, arl_diagnostics) = calibrate_cusum_thresholds(validation_increments=(validation_llr),
                                                                           validation_cusum=(validation_G),
                                                                           target_arl=160,
                                                                           num_states=10,
                                                                           candidate_offsets=range(5, 40),
                                                                           fallback_factor=2.5,
                                                                           burn_in=30)

################# CUSUM-BASED ANOMALY DETECTION ####################

(n_d, n_c, S, G) = cusum_from_increments(increments=test_llr, thresholds=G_thresholds, burn_in=30)

# Convert earliest sensor detection into system labels.
new_labels = np.zeros(test_errors.shape[0], dtype=int)

if np.any(np.isfinite(n_d)):
    first_detection = int(np.nanmin(n_d))

    new_labels[first_detection:] = 1

else:
    first_detection = None


# Final performance
macro_f1 = f1_score(original_labels, new_labels, average="macro")
mcc = matthews_corrcoef(original_labels, new_labels)


# SENSOR-BASED CUSUM SUMMARY
parameter_summary = pd.DataFrame({
    "sensor":
        np.arange(
            len(mu0_mle)
        ),

    "sensor_name":
        var_list,

    "mu0_mle":
        mu0_mle,

    "sigma_mle":
        sigma_mle,

    "mu1_mle":
        mu1_mle,

    "validation_cusum_max":
        lower_bounds,

    "selected_G_threshold":
        G_thresholds,

    "detection_time":
        n_d,

    "change_point":
        n_c})


selected_arl_rows = []
for sensor in range(len(var_list)):
    
    sensor_rows = arl_diagnostics[arl_diagnostics["sensor"]== sensor]

    matching_rows = sensor_rows[np.isclose(sensor_rows["threshold"], G_thresholds[sensor])]

    if matching_rows.empty:
        selected_arl_rows.append({
            "sensor": sensor,
            "selected_arl": np.nan,
            "threshold_source":
                "fallback"})

    else:
        selected_arl_rows.append({
            "sensor": sensor,
            "selected_arl": float(matching_rows.iloc[0]["arl0"]),
            "threshold_source":
                "Markov ARL"})


selected_arl_summary = pd.DataFrame(selected_arl_rows)

parameter_summary = (parameter_summary.merge(selected_arl_summary, on="sensor", how="left"))


# PRINT AND SAVE

markov_count = int((parameter_summary["threshold_source"]== "Markov ARL").sum())

fallback_count = int((parameter_summary["threshold_source"]== "fallback").sum())

print("\n" "=== FOLDED-NORMAL " "MLE + MARKOV CUSUM ===")

print("MLE successful for all sensors:", bool(mle_success.all()))

print("Sensors using Markov ARL threshold:",markov_count)
print("Sensors using fallback threshold:", fallback_count)

print("First system detection:", first_detection)

print("Macro-F1:", round(macro_f1, 6))
print("MCC:", round(mcc, 6))

print("\nDetection times:")
print(np.where(np.isnan(n_d), "no anomaly", n_d,))

print("\nEstimated change points:")
print(np.where(np.isnan(n_c), "no anomaly", n_c))

print("\nSensor summary:")

print(parameter_summary.to_string(index=False))

## Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# SETTINGS
scenario_name = ""
fault_onset_time = 130
burn_in = 30
number_of_sensors = test_errors.shape[1]

# INPUT ARRAYS
n_d = np.asarray(n_d, dtype=float).reshape(-1)
n_c = np.asarray(n_c, dtype=float).reshape(-1)
G_thresholds = np.asarray(G_thresholds, dtype=float).reshape(-1)


# SENSOR RESPONSIBLE FOR SYSTEM-LEVEL DETECTION
valid_detection = np.where(np.isfinite(n_d))[0]
sensor_detection = int(valid_detection[np.argmin(n_d[valid_detection])])
detection_time = int(n_d[sensor_detection])


# SENSOR RESPONSIBLE FOR SYSTEM-LEVEL CHANGE-POINT ESTIMATE
valid_change          = np.where(np.isfinite(n_c))[0]
sensor_change         = int(valid_change[np.argmin(n_c[valid_change])])
estimated_change_time = int(n_c[sensor_change])


# SENSOR INFORMATION
var_det = sensor_detection + 1
var_cp  = sensor_change + 1
u_det   = float(G_thresholds[sensor_detection])

same_sensor = bool(sensor_detection==sensor_change)

# (Optional) original sensor names
if ("var_list" in globals() and var_list is not None and len(var_list) == number_of_sensors):
    original_name_detection = str(var_list[sensor_detection])

    original_name_change = str(var_list[sensor_change])

else:
    original_name_detection = (f"sensor_{sensor_detection}")
    original_name_change = (f"sensor_{sensor_change}")

# METRICS
detection_delay = (detection_time - fault_onset_time)
change_point_error = abs(estimated_change_time - fault_onset_time)


# PRINT SUMMARY
print("=" * 72)
print(f"Scenario: {scenario_name}")

print("Detection sensor: " f"x_{var_det} " f"({original_name_detection}, " f"Python index {sensor_detection})")
print("Change-point sensor: " f"x_{var_cp} " f"({original_name_change}, " f"Python index {sensor_change})")

print(f"Same sensor: {same_sensor}")
print(f"True fault onset: n_c = {fault_onset_time}")

print(f"System-level detection time: " f"n_d^sys = {detection_time}")
print(f"System-level estimated change point: " f"n_c_hat^sys = {estimated_change_time}")
print(f"Detection delay: D_det = {detection_delay}")
print(f"Absolute change-point error: " f"E_cp = {change_point_error}")
print(f"CUSUM threshold: " f"u_{var_det} = {u_det:.6f}")

print("=" * 72)


# DISPLAY WINDOW
plot_start = max(burn_in, fault_onset_time - 60)
plot_end = min(test_errors.shape[0], max(detection_time + 25, estimated_change_time + 25, fault_onset_time + 50))

if plot_end <= plot_start:
    raise ValueError(
        "Invalid plotting interval: "
        f"plot_start={plot_start}, "
        f"plot_end={plot_end}.")

time_axis = np.arange(plot_start, plot_end)


# DISPLAYED TRAJECTORIES
A_detection_values = test_errors[plot_start:plot_end, sensor_detection]
J_detection_values = G[plot_start:plot_end, sensor_detection]

A_change_values = test_errors[plot_start:plot_end, sensor_change]
S_change_values = S[plot_start:plot_end, sensor_change]

# COLORS AND STYLES
COLOR_RESIDUAL = "#2F80B7"
COLOR_J = "#7851A9"
COLOR_S = "#B66549"

COLOR_FAULT  = "#575D63"
COLOR_CHANGE = "#238B67"
COLOR_DETECTION = "#D24A43"
COLOR_THRESHOLD = "#60435F"

GRID_COLOR = "#C7CCD1"
AXIS_COLOR = "#687078"

FAULT_LINESTYLE = (0, (2, 2))
DETECTION_LINESTYLE = (0, (6, 3))
THRESHOLD_LINESTYLE = (0, (7, 4))
CHANGE_LINESTYLE = "-."

YLABEL_SIZE = 15
XLABEL_SIZE = 12
TITLE_SIZE = 14
TICK_SIZE = 10
LEGEND_SIZE = 10


# DRAWING HELPERS
def draw_true_onset(axis):
    axis.axvline(x=fault_onset_time, color=COLOR_FAULT, linestyle=FAULT_LINESTYLE, linewidth=1.8, zorder=4)

def draw_detection_information(axis):
    draw_true_onset(axis)
    axis.axvline(x=detection_time, color=COLOR_DETECTION, linestyle=DETECTION_LINESTYLE, linewidth=2.2, zorder=6)
    axis.axhline(y=u_det, color=COLOR_THRESHOLD, linestyle=THRESHOLD_LINESTYLE, linewidth=1.7, zorder=3)

    axis.scatter(detection_time, G[detection_time, sensor_detection], s=75, color=COLOR_DETECTION, edgecolor="white", linewidth=1.1, zorder=8)
    
    # Label immediately next to the detection line.
    axis.annotate((rf"$n_d^{{\mathrm{{sys}}}}" rf"={detection_time}$"),
                  xy=(detection_time, 0.94),
                  xycoords=("data", "axes fraction"),
                  xytext=(7, 0),
                  textcoords="offset points", 
                  ha="left",
                  va="top",
                  color=COLOR_DETECTION,
                  fontsize=10,
                  fontweight="bold",
                  zorder=10)


def draw_change_point_information(axis):
    draw_true_onset(axis)

    axis.axvline(
        x=estimated_change_time,
        color=COLOR_CHANGE,
        linestyle=CHANGE_LINESTYLE,
        linewidth=2.2,
        zorder=6)

    axis.scatter(estimated_change_time, S[estimated_change_time, sensor_change],
                 s=75,
                color=COLOR_CHANGE,
                edgecolor="white",
                linewidth=1.1,
                zorder=8)

    # Label immediately next to the estimated change-point line.
    axis.annotate((rf"$\widehat{{n}}_c^{{\mathrm{{sys}}}}" rf"={estimated_change_time}$"),
                  xy=(estimated_change_time, 0.94),
                  xycoords=("data", "axes fraction"),
                  xytext=(7, 0),
                  textcoords="offset points",
                  ha="left",
                  va="top",
                  color=COLOR_CHANGE,
                  fontsize=10,
                  fontweight="bold",
                  zorder=10)


# CREATE FIGURE
if same_sensor:
    fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(13.5, 8.8), sharex=True)
    (ax_A, ax_J, ax_S) = axes


    # PANEL 1: RESIDUAL
    ax_A.plot(time_axis, A_detection_values, color=COLOR_RESIDUAL, linewidth=1.8)

    ax_A.set_ylabel(rf"$\mathcal{{A}}_{{{var_det}}}(t)$", fontsize=YLABEL_SIZE)

    ax_A.set_title(rf"{scenario_name}: sequential CUSUM behavior for " rf"$x_{{{var_det}}}$", fontsize=TITLE_SIZE, pad=10)

    # Only the true onset is shown on residual panels.
    draw_true_onset(ax_A)


    # PANEL 2: ALARM STATISTIC
    ax_J.plot(time_axis, J_detection_values, color=COLOR_J, linewidth=2.1)

    ax_J.fill_between(time_axis, J_detection_values, u_det, where=(J_detection_values >= u_det),
                      color=COLOR_DETECTION,
                      alpha=0.12,
                      interpolate=True)

    ax_J.set_ylabel(rf"$J_{{{var_det}}}(t)$", fontsize=YLABEL_SIZE)
    draw_detection_information(ax_J)


    # PANEL 3: HISTORY TRACKER
    ax_S.plot(time_axis, S_change_values, color=COLOR_S, linewidth=2.1)

    ax_S.set_ylabel(rf"$S_{{{var_cp}}}(t)$", fontsize=YLABEL_SIZE)

    ax_S.set_xlabel("Prediction-aligned time step", fontsize=XLABEL_SIZE)

    draw_change_point_information(ax_S)

else:
    fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(13.5, 11.2), sharex=True)

    (ax_A_detection, ax_J, ax_A_change, ax_S) = axes

    
    # PANEL 1: RESIDUAL OF DETECTION SENSOR
    ax_A_detection.plot(
        time_axis,
        A_detection_values,
        color=COLOR_RESIDUAL,
        linewidth=1.8)

    ax_A_detection.set_ylabel(
        rf"$\mathcal{{A}}_{{{var_det}}}(t)$",
        fontsize=YLABEL_SIZE)

    ax_A_detection.set_title(
        rf"{scenario_name}: "
        rf"$n_d^{{\mathrm{{sys}}}}$ from "
        rf"$x_{{{var_det}}}$ and "
        rf"$\widehat{{n}}_c^{{\mathrm{{sys}}}}$ from "
        rf"$x_{{{var_cp}}}$",
        fontsize=TITLE_SIZE,
        pad=10)

    # Only true onset on residual panel
    draw_true_onset(ax_A_detection)

    
    # PANEL 2: J OF DETECTION SENSOR
    ax_J.plot(time_axis, J_detection_values, color=COLOR_J, linewidth=2.1)

    ax_J.fill_between(time_axis, J_detection_values, u_det,
                      where=(J_detection_values>= u_det),
                      color=COLOR_DETECTION,
                      alpha=0.12,
                      interpolate=True)

    ax_J.set_ylabel(
        rf"$J_{{{var_det}}}(t)$",
        fontsize=YLABEL_SIZE)

    draw_detection_information(ax_J)

    
    # PANEL 3: RESIDUAL OF CHANGE-POINT SENSOR
    ax_A_change.plot(
        time_axis,
        A_change_values,
        color=COLOR_RESIDUAL,
        linewidth=1.8)

    ax_A_change.set_ylabel(
        rf"$\mathcal{{A}}_{{{var_cp}}}(t)$",
        fontsize=YLABEL_SIZE)

    # Only true onset on residual panel.
    draw_true_onset(ax_A_change)

    
    # PANEL 4: S OF CHANGE-POINT SENSOR
    ax_S.plot(
        time_axis,
        S_change_values,
        color=COLOR_S,
        linewidth=2.1)

    ax_S.set_ylabel(
        rf"$S_{{{var_cp}}}(t)$",
        fontsize=YLABEL_SIZE)

    ax_S.set_xlabel(
        "Prediction-aligned time step",
        fontsize=XLABEL_SIZE,
    )

    draw_change_point_information(ax_S)
    

# GENERAL AXIS STYLING
for axis in axes:
    axis.grid(
        True,
        which="major",
        color=GRID_COLOR,
        linestyle="--",
        linewidth=0.7,
        alpha=0.45)

    axis.set_axisbelow(True)
    axis.spines["top"].set_visible(False)

    axis.spines["right"].set_visible(False)
    axis.spines["left"].set_color(AXIS_COLOR)

    axis.spines["bottom"].set_color(AXIS_COLOR)

    axis.tick_params(
        axis="both",
        colors=AXIS_COLOR,
        labelsize=TICK_SIZE)

    axis.set_xlim(plot_start, plot_end - 1)

# COMMON BOTTOM LEGEND
legend_handles = [
    Line2D(
        [0],
        [0],
        color=COLOR_FAULT,
        linestyle=FAULT_LINESTYLE,
        linewidth=2.0,
        label=(
            rf"True fault onset: "
            rf"$n_c^{{\mathrm{{sys}}}}={fault_onset_time}$")),
    Line2D(
        [0],
        [0],
        color=COLOR_DETECTION,
        linestyle=DETECTION_LINESTYLE,
        linewidth=2.4,
        label=(
            rf"Detection time: "
            rf"$n_d^{{\mathrm{{sys}}}}"
            rf"={detection_time}$")),
    Line2D(
        [0],
        [0],
        color=COLOR_CHANGE,
        linestyle=CHANGE_LINESTYLE,
        linewidth=2.4,
        label=(
            rf"Estimated change point: "
            rf"$\widehat{{n}}_c^{{\mathrm{{sys}}}}"
            rf"={estimated_change_time}$")),
    Line2D(
        [0],
        [0],
        color=COLOR_THRESHOLD,
        linestyle=THRESHOLD_LINESTYLE,
        linewidth=2.0,
        label=(
            rf"CUSUM threshold: "
            rf"$u_{{{var_det}}}={u_det:.2f}$"))]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(
        0.5,
        0.012),
    ncol=2,
    frameon=False,
    fontsize=LEGEND_SIZE,
    handlelength=4.0,
    handletextpad=0.9,
    columnspacing=2.0)


# FINAL LAYOUT
plt.tight_layout(rect=[0, 0.105, 1, 1])
fig.subplots_adjust(hspace=0.18, bottom=0.14)


# Margin-only CUSUM-Anomaly Detection

In [ ]:
gamma_values = []
margin_results = {}
aggregate_rows = []
sensor_rows = []

# RUN MARGIN-ONLY CALIBRATION FOR EACH GAMMA
for gamma in gamma_values:
    gamma = float(gamma)
    
    # Margin-only thresholds
    (margin_thresholds, validation_cusum_max) = calibrate_margin_only_thresholds(validation_cusum=validation_G, gamma=gamma)

    # Test CUSUM
    (margin_n_d, margin_n_c, margin_S, margin_G) = cusum_from_increments(increments=test_llr, thresholds=margin_thresholds, burn_in=30)

    # System-level results

    system_result = extract_system_results(
        detection_times=margin_n_d,
        change_points=margin_n_c,
        number_of_steps=a.shape[0],
        true_labels=original_labels,
        true_change_point=130)

    detection_sensor = system_result["detection_sensor"]
    change_point_sensor = system_result["change_point_sensor"]

    # Detection sensor information
    if detection_sensor is None:
        detection_sensor_name = None
        detection_sensor_thesis_number = np.nan

    else:
        detection_sensor_name = str(var_list[detection_sensor])
        detection_sensor_thesis_number = int(detection_sensor + 1)

    # Change-point sensor information
    if change_point_sensor is None:
        change_point_sensor_name = None
        change_point_sensor_thesis_number = np.nan

    else:
        change_point_sensor_name = str(var_list[change_point_sensor])
        change_point_sensor_thesis_number = int(change_point_sensor + 1)

    # Aggregate row
    aggregate_rows.append({
        "gamma":
            gamma,

        "macro_f1":
            system_result[
                "macro_f1"
            ],

        "mcc":
            system_result[
                "mcc"
            ],

        "system_detection_time":
            system_result[
                "system_detection_time"
            ],

        "system_change_point":
            system_result[
                "system_change_point"
            ],

        "detection_delay":
            system_result[
                "detection_delay"
            ],

        "change_point_error":
            system_result[
                "change_point_error"
            ],

        "pre_change_alarm":
            system_result[
                "pre_change_alarm"
            ],

        "detection_sensor_index":
            detection_sensor,

        "detection_sensor_thesis_number":
            detection_sensor_thesis_number,

        "detection_sensor_name":
            detection_sensor_name,

        "change_point_sensor_index":
            change_point_sensor,

        "change_point_sensor_thesis_number":
            change_point_sensor_thesis_number,

        "change_point_sensor_name":
            change_point_sensor_name})


    # Sensor-level rows
    for sensor in range(number_of_sensors):
        sensor_rows.append({
            "gamma":
                gamma,

            "sensor":
                sensor,

            "thesis_variable_number":
                sensor + 1,

            "sensor_name":
                str(
                    var_list[sensor]
                ),

            "mu0_mle":
                mu0_mle[
                    sensor
                ],

            "sigma_mle":
                sigma_mle[
                    sensor
                ],

            "mu1_mle":
                mu1_mle[
                    sensor
                ],

            "pointwise_shift_tau":
                sensor_thresholds[
                    sensor
                ],

            "validation_cusum_max":
                validation_cusum_max[
                    sensor
                ],

            "margin_threshold":
                margin_thresholds[
                    sensor
                ],

            "detection_time":
                margin_n_d[
                    sensor
                ],

            "change_point":
                margin_n_c[
                    sensor
                ]})
        
        


# RESULT SUMMARIES
margin_aggregate_summary = pd.DataFrame(aggregate_rows)
margin_sensor_summary = pd.DataFrame(sensor_rows)

# PRINT RESULTS
print(
    "\n"
    "=== FOLDED-NORMAL MLE + "
    "MARGIN-ONLY CUSUM ===")

print("MLE successful for all sensors:", bool(mle_success.all()))

print("\nAggregate margin-only results:")

print(margin_aggregate_summary.to_string(index=False))

for gamma in gamma_values:
    gamma = float(gamma)
    result = margin_results[gamma]

    print("\n" + "=" * 72)

    print(f"MARGIN-ONLY CALIBRATION: gamma={gamma}")
    print("=" * 72)

    print("Macro-F1:", round(result["macro_f1"], 6))
    print("MCC:", round(result["mcc"], 6))

    print("System-level detection time:", result["system_detection_time"])

    print("System-level estimated change point:", result["system_change_point"])

    print("Detection delay:", result["detection_delay"])

    print("Absolute change-point error:", result["change_point_error"])

    print("Pre-change alarm:", result["pre_change_alarm"])

    print("Detection sensor index:", result["detection_sensor"])

    print("Change-point sensor index:", result["change_point_sensor"])

    print("\nSensor-specific detection times:")

    print(np.where(np.isnan(result["n_d"]), "no anomaly", result["n_d"].astype(object)))

    print("\nSensor-specific estimated change points:")
    print(np.where(np.isnan(result["n_c"]), "no anomaly", result["n_c"].astype(object)))

